In [3]:
import json
import os
import mesa
from google import genai
from google.genai import types

api_key = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

class SpatialAgent(mesa.Agent):
    def __init__(self, model, persona):
        super().__init__(model)
        self.persona = persona

    def step(self):
        # 1. Gather Spatial Context from Mesa's Grid
        neighbors = self.model.grid.get_neighborhood(self.pos, moore=True, include_center=False)
        other_agents = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
        agent_locs = {a.unique_id: a.pos for a in other_agents}

        # 2. Construct the Spatial Prompt
        prompt = f"""
        You are Agent {self.unique_id}. Persona: {self.persona}.
        Your current grid coordinates: {self.pos}.
        Valid adjacent coordinates you can move to: {neighbors}.
        Other agents currently in your visual range: {agent_locs if agent_locs else "None"}.

        Respond STRICTLY with JSON containing your movement choice and your reasoning:
        {{
            "new_pos": [x, y],
            "dialogue": "your response in English explaining why you moved here"
        }}
        *CRITICAL: "new_pos" MUST be an exact [x, y] coordinate array chosen from your valid adjacent coordinates.*
        """

        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                ),
            )
            decision = json.loads(response.text)
            
            new_pos_list = decision.get("new_pos")
            dialogue = decision.get("dialogue")
            
            # 3. Execute Physical Movement on the Grid
            if new_pos_list:
                new_pos = tuple(new_pos_list)
                if new_pos in neighbors:
                    self.model.grid.move_agent(self, new_pos)
                    print(f"Agent {self.unique_id} ({self.persona}) MOVED to {new_pos}: '{dialogue}'")
                else:
                    print(f"Agent {self.unique_id} hallucinated an invalid coordinate {new_pos}. Stayed put.")
            
        except Exception as e:
            print(f"Error for Agent {self.unique_id}: {e}")

class SpatialModel(mesa.Model):
    def __init__(self, width=5, height=5):
        super().__init__()
        self.grid = mesa.space.MultiGrid(width, height, True)
        
        self.agent1 = SpatialAgent(self, "Introvert seeking empty space away from others")
        self.agent2 = SpatialAgent(self, "Extrovert trying to get as close to other agents as possible")
        
        self.grid.place_agent(self.agent1, (1, 1))
        self.grid.place_agent(self.agent2, (3, 3))

    def step(self):
        self.agent1.step()
        self.agent2.step()

In [4]:
spatial_model = SpatialModel()
for i in range(3):
    print(f"\n--- STEP {i+1} ---")
    spatial_model.step()


--- STEP 1 ---
Agent 1 (Introvert seeking empty space away from others) MOVED to (0, 0): 'Moving to a new spot to ensure continued solitude. With no one else around, any space is a good space to settle into.'
Agent 2 (Extrovert trying to get as close to other agents as possible) MOVED to (3, 4): 'Time to explore and see who's out and about! A little stroll might help me bump into some friendly faces.'

--- STEP 2 ---
Agent 1 (Introvert seeking empty space away from others) MOVED to (0, 1): 'No one is around, so any move will do. I'll take a step to (0,1), just to put some more distance between 'here' and 'there'.'
Agent 2 (Extrovert trying to get as close to other agents as possible) MOVED to (3, 3): 'No one in sight yet! That's okay, I'll just move a little closer to the heart of things. Can't wait to meet some new faces!'

--- STEP 3 ---
Agent 1 (Introvert seeking empty space away from others) MOVED to (1, 1): 'As an introvert, my goal is to find space away from others. Since there 